# Inventory Optimization — Forecasted Demand

Computes **Safety Stock**, **Reorder Point (ROP)**, and **EOQ** per `(Store ID, Product ID)`  
using `retail_store_inventory.csv`. Flags each SKU as `STOCKOUT_RISK`, `OVERSTOCK`, or `OPTIMAL`.

For the full production run (MLflow logging + CSV output):  
```
python run_inventory.py --service-level 0.95 --lead-time 7
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.inventory_optimizer import (
    InventoryOptimizer,
    STATUS_OPTIMAL,
    STATUS_OVERSTOCK,
    STATUS_STOCKOUT,
    INVENTORY_FILE,
)

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

## 1. Load & Inspect Raw Inventory Data

In [ ]:
df = pd.read_csv(INVENTORY_FILE, parse_dates=['Date'])
df.columns = df.columns.str.strip()

print(f'Rows          : {len(df):,}')
print(f'Date range    : {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Stores        : {df["Store ID"].nunique()}')
print(f'Products      : {df["Product ID"].nunique()}')
print(f'Categories    : {df["Category"].nunique()}  -> {sorted(df["Category"].unique())}')
print(f'Regions       : {df["Region"].nunique()}    -> {sorted(df["Region"].unique())}')
df.head()

In [ ]:
print('=== Units Sold ===')
print(df['Units Sold'].describe().round(2))
print(f'Zeros: {(df["Units Sold"] == 0).sum()}')
print()
print('=== Demand Forecast ===')
print(df['Demand Forecast'].describe().round(2))
print(f'Negatives: {(df["Demand Forecast"] < 0).sum()}')
print()
print('=== Inventory Level ===')
print(df['Inventory Level'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['Units Sold'], bins=40, color='steelblue', alpha=0.8)
axes[0].set_title('Units Sold Distribution')
axes[0].set_xlabel('Units Sold')

axes[1].hist(df['Demand Forecast'].clip(lower=0), bins=40, color='darkorange', alpha=0.8)
axes[1].set_title('Demand Forecast Distribution (clipped ≥ 0)')
axes[1].set_xlabel('Demand Forecast')

axes[2].hist(df['Inventory Level'], bins=40, color='seagreen', alpha=0.8)
axes[2].set_title('Inventory Level Distribution')
axes[2].set_xlabel('Inventory Level')

plt.tight_layout()
plt.show()

## 2. Demand Statistics per SKU

- `mean_daily_demand` — from `Demand Forecast` column, clipped to ≥ 0 (forward-looking signal)
- `std_daily_demand`  — from `Units Sold` (actual historical variability)
- `cv`               — coefficient of variation (std / mean)

In [ ]:
opt = InventoryOptimizer(service_level=0.95, lead_time_days=7)
opt.load()
stats = opt.compute_demand_stats()

print(f'SKUs: {len(stats):,}')
print()
stats[['mean_daily_demand', 'std_daily_demand', 'cv', 'avg_price']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(stats['mean_daily_demand'], bins=40, color='steelblue', alpha=0.8)
axes[0].set_title('Mean Daily Demand per SKU')
axes[0].set_xlabel('Mean Demand (units/day)')

axes[1].hist(stats['std_daily_demand'], bins=40, color='darkorange', alpha=0.8)
axes[1].set_title('Demand Std Dev per SKU')
axes[1].set_xlabel('Std Dev (units/day)')

axes[2].hist(stats['cv'].clip(upper=stats['cv'].quantile(0.99)), bins=40, color='mediumpurple', alpha=0.8)
axes[2].set_title('Coefficient of Variation (CV)')
axes[2].set_xlabel('CV')

plt.tight_layout()
plt.show()

In [ ]:
print('Mean demand by Category:')
print(stats.groupby('Category')['mean_daily_demand'].agg(['mean', 'median', 'std']).round(2))

## 3. Safety Stock & Reorder Point

**Safety Stock** = Z × σ_demand × √(lead_time)  
**ROP** = (mean_demand × lead_time) + safety_stock

In [ ]:
opt.compute_safety_stock()
opt.compute_reorder_point()
stats = opt._stats

print(f'Z-score at 95% service level: {opt._z:.4f}')
print()
stats[['safety_stock', 'rop']].describe().round(2)

In [ ]:
# Sensitivity: how safety stock changes with service level
from scipy.stats import norm

service_levels = [0.80, 0.85, 0.90, 0.95, 0.99]
avg_ss = []
for sl in service_levels:
    z = norm.ppf(sl)
    ss = z * stats['std_daily_demand'] * np.sqrt(7)
    avg_ss.append(ss.mean())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([f'{int(s*100)}%' for s in service_levels], avg_ss, marker='o', color='steelblue')
ax.axvline(x=3, color='red', linestyle='--', label='Default (95%)')
ax.set_xlabel('Service Level')
ax.set_ylabel('Avg Safety Stock (units)')
ax.set_title('Safety Stock vs Service Level')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Economic Order Quantity (EOQ)

**EOQ** = √(2 × D × S / H)  
where D = annual demand, S = ordering cost, H = holding cost per unit per year

In [ ]:
opt.compute_eoq()
stats = opt._stats

print(f'Ordering cost     : ${opt.ordering_cost}')
print(f'Holding cost rate : {opt.holding_cost_rate*100:.0f}% of unit price')
print()
stats['eoq'].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(stats['eoq'].clip(upper=stats['eoq'].quantile(0.99)), bins=40, color='steelblue', alpha=0.8)
axes[0].set_title('EOQ Distribution (99th pct clip)')
axes[0].set_xlabel('EOQ (units)')

eoq_by_cat = stats.groupby('Category')['eoq'].median().sort_values(ascending=False)
axes[1].bar(eoq_by_cat.index, eoq_by_cat.values, color='steelblue', alpha=0.8)
axes[1].set_title('Median EOQ by Category')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Median EOQ (units)')
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.show()

## 5. Current Inventory Status Evaluation

Snapshot at global max date (`2024-01-01`).  
- **STOCKOUT_RISK** — Inventory < ROP  
- **OVERSTOCK**     — Inventory > ROP + EOQ  
- **OPTIMAL**       — otherwise

In [ ]:
opt.evaluate_current_stock()
rec = opt.generate_recommendations()

print(f'Snapshot date  : {opt._df["Date"].max().date()}')
print(f'Total SKUs     : {len(rec):,}')
print()
status_counts = rec['status'].value_counts()
for s, n in status_counts.items():
    print(f'  {s:<16}: {n:>5}  ({n/len(rec)*100:.1f}%)')

In [ ]:
print('Summary statistics — recommendations:')
rec[['mean_daily_demand', 'safety_stock', 'rop', 'eoq',
     'current_inventory', 'days_of_stock', 'units_to_order']].describe().round(2)

## 6. Visualizations

In [ ]:
# Plot 1 — Stockout risk heatmap
pivot = (
    rec[rec['status'] == STATUS_STOCKOUT]
    .groupby(['store_id', 'category'])
    .size()
    .unstack(fill_value=0)
)
total_pivot = rec.groupby(['store_id', 'category']).size().unstack(fill_value=0)
pct_pivot   = (pivot / total_pivot.replace(0, np.nan) * 100).fillna(0)

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(pct_pivot, annot=True, fmt='.0f', cmap='Reds',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% SKUs at Stockout Risk'})
ax.set_title('Stockout Risk Heatmap (Store × Category)')
ax.set_xlabel('Category')
ax.set_ylabel('Store ID')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2 — EOQ vs actual Units Ordered
snap = opt._snapshot.copy()
snap = snap[snap['eoq'] > 0]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(snap['eoq'], snap['Units Ordered'], alpha=0.25, s=10, color='steelblue')
max_val = max(snap['eoq'].max(), snap['Units Ordered'].max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.2, label='EOQ = Ordered')
ax.set_xlabel('EOQ (optimal order qty)')
ax.set_ylabel('Actual Units Ordered')
ax.set_title('EOQ vs Actual Units Ordered')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3 — Safety stock distribution by category
fig, ax = plt.subplots(figsize=(10, 5))
for cat in sorted(stats['Category'].unique()):
    subset = stats[stats['Category'] == cat]['safety_stock']
    ax.hist(subset, bins=30, alpha=0.5, label=cat)
ax.set_xlabel('Safety Stock (units)')
ax.set_ylabel('SKU count')
ax.set_title('Safety Stock Distribution by Category')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4 — Days of stock remaining by category
rec_plot = rec.copy()
rec_plot['days_of_stock_plot'] = rec_plot['days_of_stock'].replace(np.inf, np.nan)
categories = sorted(rec_plot['category'].unique())
data_by_cat = [
    rec_plot[rec_plot['category'] == c]['days_of_stock_plot'].dropna().values
    for c in categories
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(data_by_cat, labels=categories, patch_artist=True)
ax.axhline(opt.lead_time_days, color='red', linestyle='--',
           linewidth=1.2, label=f'Lead time ({opt.lead_time_days}d)')
ax.set_xlabel('Category')
ax.set_ylabel('Days of Stock Remaining')
ax.set_title('Days of Stock Remaining by Category')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 5 — Inventory status breakdown by category
breakdown = (
    rec.groupby(['category', 'status'])
    .size()
    .unstack(fill_value=0)
)
for col in [STATUS_STOCKOUT, STATUS_OVERSTOCK, STATUS_OPTIMAL]:
    if col not in breakdown.columns:
        breakdown[col] = 0
breakdown = breakdown[[STATUS_OPTIMAL, STATUS_OVERSTOCK, STATUS_STOCKOUT]]

fig, ax = plt.subplots(figsize=(10, 5))
breakdown.plot(kind='bar', stacked=True, ax=ax,
               color=['#4caf50', '#ff9800', '#f44336'])
ax.set_xlabel('Category')
ax.set_ylabel('SKU count')
ax.set_title('Inventory Status Breakdown by Category')
ax.legend(title='Status', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 7. Summary Statistics Table

In [ ]:
total = len(rec)
finite_days = rec['days_of_stock'].replace(np.inf, np.nan)

summary = pd.DataFrame([
    {'Metric': 'Total SKUs evaluated',       'Value': f'{total:,}'},
    {'Metric': 'STOCKOUT_RISK (%)',           'Value': f'{(rec["status"]==STATUS_STOCKOUT).mean()*100:.1f}%'},
    {'Metric': 'OVERSTOCK (%)',               'Value': f'{(rec["status"]==STATUS_OVERSTOCK).mean()*100:.1f}%'},
    {'Metric': 'OPTIMAL (%)',                 'Value': f'{(rec["status"]==STATUS_OPTIMAL).mean()*100:.1f}%'},
    {'Metric': 'Avg days of stock',           'Value': f'{finite_days.mean():.1f}'},
    {'Metric': 'Median days of stock',        'Value': f'{finite_days.median():.1f}'},
    {'Metric': 'Total units to order',        'Value': f'{int(rec["units_to_order"].sum()):,}'},
    {'Metric': 'Avg safety stock (units)',    'Value': f'{rec["safety_stock"].mean():.1f}'},
    {'Metric': 'Avg EOQ (units)',             'Value': f'{rec["eoq"].mean():.1f}'},
    {'Metric': 'Avg ROP (units)',             'Value': f'{rec["rop"].mean():.1f}'},
    {'Metric': 'Service level',               'Value': f'{opt.service_level*100:.0f}%'},
    {'Metric': 'Lead time (days)',            'Value': str(opt.lead_time_days)},
])
summary

## 8. Full Production Run

Saves charts to `reports/figures/`, writes `data/processed/inventory_recommendations.csv`,  
and logs everything to MLflow under the `inventory_optimization` experiment.

In [ ]:
opt_prod = InventoryOptimizer(service_level=0.95, lead_time_days=7,
                               ordering_cost=50.0, holding_cost_rate=0.25)
recommendations = opt_prod.run()
print(f'\nRecommendations saved: {len(recommendations):,} rows')
recommendations.head(10)